<a href="https://colab.research.google.com/github/DangHuuLong/Ai-Recruiter-Mini-Ai-Service/blob/experiment%2Fcross-encoder-v0.2/notebooks/fine_tune_cross_encoder_v0_2_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tune Cross-Encoder CV-JD v0.2

Fine-tune `cross-encoder/ms-marco-MiniLM-L-12-v2` (12 layers) trên 4900 CV-JD pairs.
So sánh với v0.1 (`MiniLM-L-6-v2`, 6 layers) — mục tiêu cải thiện LabelAcc vượt 58%.

| | |
|---|---|
| **Dataset** | v0.3 — 4900 train / 1050 validation / 1050 test |
| **Base model** | `cross-encoder/ms-marco-MiniLM-L-12-v2` |
| **Loss** | MSE regression, label = score / 100, sigmoid activation |
| **max_length** | 512 tokens |
| **Branch** | `experiment/cross-encoder-v0.2` |


In [1]:
import os
if not os.path.exists('/content/Ai-Recruiter-Mini-Ai-Service'):
    !git clone https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service /content/Ai-Recruiter-Mini-Ai-Service

%cd /content/Ai-Recruiter-Mini-Ai-Service
!git checkout experiment/cross-encoder-v0.2
!git pull origin experiment/cross-encoder-v0.2


/content/Ai-Recruiter-Mini-Ai-Service
Already on 'experiment/cross-encoder-v0.2'
Your branch is ahead of 'origin/experiment/cross-encoder-v0.2' by 6 commits.
  (use "git push" to publish your local commits)
From https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service
 * branch            experiment/cross-encoder-v0.2 -> FETCH_HEAD
Already up to date.


In [25]:
!pip install -r requirements.txt


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 11.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of huggingface-hub to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 517.7/517.7 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.0/472.0 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.2/

In [2]:
from pathlib import Path
import json

data_dir = Path("datasets/versions/v0.3/cross_encoder")
for split in ("train", "validation", "test"):
    path = data_dir / f"cross_encoder_{split}.jsonl"
    lines = path.read_text(encoding="utf-8").strip().splitlines()
    first = json.loads(lines[0])
    print(f"{split:<12}: {len(lines):>5} pairs  | keys: {list(first.keys())}")


train       :  4900 pairs  | keys: ['pair_id', 'cv_text', 'jd_text', 'score', 'label', 'true_label']
validation  :  1050 pairs  | keys: ['pair_id', 'cv_text', 'jd_text', 'score', 'label', 'true_label']
test        :  1050 pairs  | keys: ['pair_id', 'cv_text', 'jd_text', 'score', 'label', 'true_label']


In [3]:
!WANDB_MODE=disabled python -m training.fine_tune_cross_encoder \
    --base-model cross-encoder/ms-marco-MiniLM-L-12-v2 \
    --epochs 1 \
    --batch-size 4 \
    --max-train-samples 40 \
    --max-eval-samples 20


2026-06-18 11:02:09.653047: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Base model : cross-encoder/ms-marco-MiniLM-L-12-v2
Data dir   : datasets/versions/v0.3/cross_encoder
Output dir : artifacts/models/cross-encoder-cv-jd-v0.1
Epochs     : 1  |  Batch size: 4  |  Max length: 512

Train: 40 pairs  |  Val: 20 pairs

Steps/epoch: 10  |  Warmup steps: 1

Epoch:   0% 0/1 [00:00<?, ?it/s]
Iteration:   0% 0/10 [00:00<?, ?it/s]
Iteration:  10% 1/10 [00:00<00:05,  1.63it/s]
Iteration:  30% 3/10 [00:00<00:01,  4.19it/s]
Iteration:  50% 5/10 [00:00<00:00,  5.77it/s]
Iteration:  70% 7/10 [00:01<00:00,  7.03it/s]
Iteration: 100% 10/10 [00:01<00:00,  7.21it/s]
Epoch: 100% 1/1 [00:01<00:00,  1.88s/it]

Fine-tuning complete.
Model written to: artifacts/mode

In [4]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


CUDA available: True
Device: Tesla T4


In [6]:
from google.colab import drive
drive.mount('/content/drive')

drive_base = "/content/drive/MyDrive/ai-recruiter"

# Tạo thư mục trước để Drive có thời gian sync
import os, time
os.makedirs(f"{drive_base}/models/cross-encoder-cv-jd-v0.2", exist_ok=True)
time.sleep(3)

!WANDB_MODE=disabled python -m training.fine_tune_cross_encoder \
    --base-model cross-encoder/ms-marco-MiniLM-L-12-v2 \
    --output-dir {drive_base}/models/cross-encoder-cv-jd-v0.2 \
    --report-path artifacts/reports/fine_tune_cross_encoder_v0.2_report.json \
    --epochs 10 \
    --batch-size 16


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2026-06-18 11:10:17.609735: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Base model : cross-encoder/ms-marco-MiniLM-L-12-v2
Data dir   : datasets/versions/v0.3/cross_encoder
Output dir : /content/drive/MyDrive/ai-recruiter/models/cross-encoder-cv-jd-v0.2
Epochs     : 10  |  Batch size: 16  |  Max length: 512

Train: 4900 pairs  |  Val: 1050 pairs

Steps/epoch: 307  |  Warmup steps: 30

Epoch:   0% 0/10 [00:00<?, ?it/s]
Iteration:   0% 0/307 [00:00<?, ?it/s]
Iteration:   0% 1/307 [00:00<04:54,  1.04it/s]
Iteration:   1% 3/307 [00:01<02:32,  1.99it/s]
Iteration:   1% 4/307 [00:02<02:33,  1.98it/s]
Iteration:   2% 5/307

In [7]:
import json
from pathlib import Path

report = json.loads(
    Path("artifacts/reports/fine_tune_cross_encoder_v0.2_report.json").read_text(encoding="utf-8")
)
print(json.dumps(report["metrics"], indent=2))


{
  "validation": {
    "mae": 10.1183,
    "rmse": 13.858,
    "label_accuracy": 0.5533,
    "pair_count": 1050,
    "mean_predicted_score": 63.8533,
    "mean_target_score": 60.6457
  },
  "test": {
    "mae": 9.5019,
    "rmse": 13.5925,
    "label_accuracy": 0.6086,
    "pair_count": 1050,
    "mean_predicted_score": 64.0089,
    "mean_target_score": 62.8514
  }
}


In [8]:
import shutil
from pathlib import Path

reports_dir = Path(drive_base) / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
shutil.copy(
    "artifacts/reports/fine_tune_cross_encoder_v0.2_report.json",
    reports_dir / "fine_tune_cross_encoder_v0.2_report.json",
)
print(f"Saved to {reports_dir}")


Saved to /content/drive/MyDrive/ai-recruiter/reports
